<a href="https://colab.research.google.com/github/tanishqgandhi1908/Profiling-the-Internal-State-of-LLM-Inference/blob/main/Llama_1B_Pearson_Correlation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch numpy transformers accelerate huggingface_hub datasets

In [ ]:
import os

# --- CONFIGURATION ---
MODEL_ID = "meta-llama/Llama-3.2-1B"
CORRELATION_THRESHOLD = 0.90         # Lowered slightly to 0.90 to catch more near-redundancies!
NUM_QUESTIONS = 100                  # Number of NQ questions to process

def main():
    # --- COLAB AUTHENTICATION SETUP ---
    try:
        from google.colab import userdata
        from huggingface_hub import login
        print("Detected Google Colab! Looking for HF_TOKEN in secrets...")
        try:
            hf_token = userdata.get('HF_TOKEN')
            login(token=hf_token)
            print("Successfully logged into Hugging Face!")
        except userdata.SecretNotFoundError:
            print("\n[ERROR] 'HF_TOKEN' secret not found!")
            print("1. Click the 'Key' icon on the left sidebar in Colab.")
            print("2. Add a new secret named HF_TOKEN and paste your Hugging Face token.")
            print("3. Toggle 'Notebook access' on and run this cell again.\n")
            return
    except ImportError:
        pass

    print(f"\nLoading {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="cuda", torch_dtype=torch.float16)

    # 1. Set up the Hooks for EVERY layer
    num_layers = len(model.model.layers)
    z_vectors_per_layer = {i: [] for i in range(num_layers)}
    hooks = []

    # Helper function to create a layer-specific hook
    def make_hook(layer_idx):
        def capture_z_vector_hook(module, input, output):
            # Move to CPU immediately to prevent GPU Out-of-Memory errors
            z_vectors_per_layer[layer_idx].append(output.detach().cpu().float())
        return capture_z_vector_hook

    print(f"Attaching hooks to all {num_layers} layers...")
    for i in range(num_layers):
        target_module = model.model.layers[i].mlp
        hook = target_module.register_forward_hook(make_hook(i))
        hooks.append(hook)

    # 2. Load the NQ-Open Dataset
    print("\nDownloading NQ-Open dataset...")
    # We grab the first NUM_QUESTIONS from the training split
    dataset = load_dataset("nq_open", split=f"train[:{NUM_QUESTIONS}]")

    # 3. Run Inference
    print(f"Running forward passes on {NUM_QUESTIONS} questions...")

    with torch.no_grad():
        for item in dataset:
            # Format the question into a standard prompt
            text = f"Question: {item['question']}\nAnswer:"
            inputs = tokenizer(text, return_tensors="pt").to(model.device)

            # We use a standard forward pass instead of generate()!
            # This instantly processes all tokens in the prompt at once,
            # triggering the hook for every single token simultaneously.
            model(**inputs)

    # Clean up the hooks
    for hook in hooks:
        hook.remove()

    # 4. Process and Calculate Pearson Correlation for EVERY layer
    print("\n" + "="*50)
    print("Calculating Pearson Correlation Matrices...")

    for layer_idx in range(num_layers):
        print(f"\n--- Layer {layer_idx} ---")

        # z_vectors_per_layer[layer_idx] is a list of tensors with shape (1, seq_len, hidden_dim)
        # We reshape each to (seq_len, hidden_dim) and stack them vertically
        layer_z_vectors = z_vectors_per_layer[layer_idx]
        if not layer_z_vectors:
            continue

        Z_matrix = torch.cat([z.view(-1, z.shape[-1]) for z in layer_z_vectors], dim=0)
        total_tokens, hidden_dim = Z_matrix.shape

        # Calculate Correlation Matrix
        # Must transpose so variables (neurons) are rows and observations (tokens) are columns
        correlation_matrix = torch.corrcoef(Z_matrix.T)

        # Identify Highly Correlated Pairs
        upper_tri_indices = torch.triu_indices(row=hidden_dim, col=hidden_dim, offset=1)
        correlations = correlation_matrix[upper_tri_indices[0], upper_tri_indices[1]]

        # Remove NaN values (sometimes caused by neurons that output perfectly constant values like 0)
        valid_mask = ~torch.isnan(correlations)
        upper_tri_indices_0 = upper_tri_indices[0][valid_mask]
        upper_tri_indices_1 = upper_tri_indices[1][valid_mask]
        correlations = correlations[valid_mask]

        high_corr_mask = torch.abs(correlations) > CORRELATION_THRESHOLD

        neuron_a_indices = upper_tri_indices_0[high_corr_mask]
        neuron_b_indices = upper_tri_indices_1[high_corr_mask]
        high_corr_values = correlations[high_corr_mask]

        found_pairs = len(high_corr_values)

        if found_pairs > 0:
            print(f"Found {found_pairs} highly correlated pairs! (Shape: {total_tokens}x{hidden_dim})")
            sorted_indices = torch.argsort(torch.abs(high_corr_values), descending=True)
            print(f"Top Redundancies (Threshold: {CORRELATION_THRESHOLD}):")

            for i in range(min(5, found_pairs)): # Show max 5 per layer
                idx = sorted_indices[i]
                n_a = neuron_a_indices[idx].item()
                n_b = neuron_b_indices[idx].item()
                r_val = high_corr_values[idx].item()
                print(f"  Neuron {n_a:4d} & Neuron {n_b:4d}  |  r = {r_val:.4f}")
        else:
            print(f"No pairs found > {CORRELATION_THRESHOLD}")

if __name__ == "__main__":
    main()

Detected Google Colab! Looking for HF_TOKEN in secrets...
Successfully logged into Hugging Face!

Loading meta-llama/Llama-3.2-1B...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Attaching hooks to all 16 layers...

Running forward passes on 100 questions...

Calculating Pearson Correlation Matrices...

--- Layer 0 ---
Found 142 highly correlated pairs! (Shape: 1620x2048)
Top Redundancies (Threshold: 0.9):
  Neuron  894 & Neuron 1703  |  r = 0.9999
  Neuron  636 & Neuron 1703  |  r = -0.9999
  Neuron 1002 & Neuron 1703  |  r = 0.9999
  Neuron  636 & Neuron  894  |  r = -0.9999
  Neuron  762 & Neuron 1703  |  r = 0.9999

--- Layer 1 ---
Found 328144 highly correlated pairs! (Shape: 1620x2048)
Top Redundancies (Threshold: 0.9):
  Neuron 1159 & Neuron 2029  |  r = 1.0000
  Neuron  400 & Neuron  698  |  r = -1.0000
  Neuron  698 & Neuron 2029  |  r = 1.0000
  Neuron  698 & Neuron 1159  |  r = 1.0000
  Neuron  400 & Neuron 2029  |  r = -1.0000

--- Layer 2 ---
No pairs found > 0.9

--- Layer 3 ---
No pairs found > 0.9

--- Layer 4 ---
No pairs found > 0.9

--- Layer 5 ---
No pairs found > 0.9

--- Layer 6 ---
No pairs found > 0.9

--- Layer 7 ---
No pairs found > 0.